In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix, precision_recall_curve
from sklearn.preprocessing import LabelEncoder
import joblib

file_path = '/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv'

full_trans = pd.read_csv(file_path)

d_columns = ['D1','D2','D3','D4','D5','D6','D7','D8','D9','D10','D11','D12','D13','D14','D15']

df = full_trans[['TransactionID', 'isFraud', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 
                  'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain', 'TransactionAmt', 'TransactionDT'] + d_columns].copy()

df['d_count'] = df[d_columns].notna().sum(axis=1)

card1_counts = df['card1'].value_counts().to_dict()
df['card1_freq'] = df['card1'].map(card1_counts)

card4_counts = df['card4'].value_counts().to_dict()
df['card4_freq'] = df['card4'].map(card4_counts)

product_counts = df['ProductCD'].value_counts().to_dict()
df['product_freq'] = df['ProductCD'].map(product_counts)

df['amt_dollars'] = df['TransactionAmt'].astype(int)
df['amt_cents'] = ((df['TransactionAmt'] - df['amt_dollars']) * 100).astype(int)

card1_amt_mean = df.groupby('card1')['TransactionAmt'].mean().to_dict()
df['card1_amt_mean'] = df['card1'].map(card1_amt_mean)

card1_amt_std = df.groupby('card1')['TransactionAmt'].std().to_dict()
df['card1_amt_std'] = df['card1'].map(card1_amt_std)

card1_tx_count = df.groupby('card1')['TransactionID'].count().to_dict()
df['card1_tx_count'] = df['card1'].map(card1_tx_count)

addr1_counts = df['addr1'].value_counts().to_dict()
df['addr1_freq'] = df['addr1'].map(addr1_counts)

addr2_counts = df['addr2'].value_counts().to_dict()
df['addr2_freq'] = df['addr2'].map(addr2_counts)

card2_counts = df['card2'].fillna(-1).value_counts().to_dict()
df['card2_freq'] = df['card2'].fillna(-1).map(card2_counts)

card3_counts = df['card3'].fillna(-1).value_counts().to_dict()
df['card3_freq'] = df['card3'].fillna(-1).map(card3_counts)

pemail_counts = df['P_emaildomain'].fillna('missing').value_counts().to_dict()
df['pemail_freq'] = df['P_emaildomain'].fillna('missing').map(pemail_counts)

remail_counts = df['R_emaildomain'].fillna('missing').value_counts().to_dict()
df['remail_freq'] = df['R_emaildomain'].fillna('missing').map(remail_counts)

df['P_email_missing'] = df['P_emaildomain'].isna().astype(int)
df['R_email_missing'] = df['R_emaildomain'].isna().astype(int)
df['same_email_domain'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)

print("Feature engineering complete. Shape:", df.shape)

Feature engineering complete. Shape: (590540, 46)


In [3]:
v_columns = [f'V{i}' for i in range(1, 340)]
train_v = pd.read_csv(file_path, usecols=v_columns)

X = pd.concat([df[['card1_freq', 'card4_freq', 'product_freq', 'amt_dollars', 'amt_cents',
                   'card1_amt_mean', 'card1_amt_std', 'card1_tx_count', 'd_count',
                   'addr1_freq', 'addr2_freq', 'card2_freq', 'card3_freq',
                   'pemail_freq', 'remail_freq', 'P_email_missing', 'R_email_missing',
                   'same_email_domain']], train_v], axis=1)

y = df['isFraud']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = XGBClassifier(n_estimators=100, max_depth=13, learning_rate=0.1, scale_pos_weight=28, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

print(f"Recall: {recall_score(y_val, y_pred):.4f}")
print(f"Precision: {precision_score(y_val, y_pred):.4f}")
print(f"F1: {f1_score(y_val, y_pred):.4f}")

Recall: 0.7716
Precision: 0.3914
F1: 0.5193


In [11]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_proba)

target_recall = 0.70
idx = (recalls >= target_recall).argmax()

print(f"At recall >= {target_recall:.0%}:")
print(f"  Threshold: {thresholds[idx]:.4f}")
print(f"  Precision: {precisions[idx]:.4f}")
print(f"  Exact recall: {recalls[idx]:.4f}")

print("\nNearby options:")
for i in range(max(0, idx-3), min(len(thresholds), idx+4)):
    print(f"  threshold={thresholds[i]:.4f}, recall={recalls[i]:.4f}, precision={precisions[i]:.4f}")

At recall >= 70%:
  Threshold: 0.0001
  Precision: 0.0350
  Exact recall: 1.0000

Nearby options:
  threshold=0.0001, recall=1.0000, precision=0.0350
  threshold=0.0001, recall=1.0000, precision=0.0350
  threshold=0.0001, recall=1.0000, precision=0.0350
  threshold=0.0001, recall=1.0000, precision=0.0350


In [13]:
print("Model type:", type(model))


Model type: <class 'xgboost.sklearn.XGBClassifier'>


In [14]:
print("y_pred_proba statistics:")
print(f"Min: {y_pred_proba.min():.6f}")
print(f"Max: {y_pred_proba.max():.6f}")
print(f"Mean: {y_pred_proba.mean():.6f}")
print(f"Median: {np.median(y_pred_proba):.6f}")

print("\nSample values (first 20):")
print(y_pred_proba[:20])

y_pred_proba statistics:
Min: 0.000107
Max: 0.999901
Mean: 0.152656
Median: 0.073337

Sample values (first 20):
[0.03022237 0.2327883  0.12731859 0.02158027 0.12472992 0.37310907
 0.1600347  0.00597263 0.00627072 0.0937342  0.08498449 0.23687291
 0.31887013 0.10316764 0.19522156 0.06603782 0.04917821 0.14463547
 0.24010064 0.49349612]


In [16]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_proba)

print("Lengths:", len(precisions), len(recalls), len(thresholds))

results = pd.DataFrame({
    'precision': precisions[:-1],
    'recall': recalls[:-1],
    'threshold': thresholds
})

print("\nPrecision at different recall levels:")
for target_recall in [0.80, 0.77, 0.75, 0.72, 0.70, 0.68, 0.65]:
    closest = results.iloc[(results['recall'] - target_recall).abs().argsort()[:1]]
    row = closest.iloc[0]
    print(f"Recall ~{target_recall:.2f}: threshold={row['threshold']:.4f}, precision={row['precision']:.4f}")

print("\nBest F1 score:")
best_f1_idx = (2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1])).argmax()
print(f"Threshold={thresholds[best_f1_idx]:.4f}, Recall={recalls[best_f1_idx]:.4f}, Precision={precisions[best_f1_idx]:.4f}, F1={2*precisions[best_f1_idx]*recalls[best_f1_idx]/(precisions[best_f1_idx]+recalls[best_f1_idx]):.4f}")

Lengths: 115779 115779 115778

Precision at different recall levels:
Recall ~0.80: threshold=0.4390, precision=0.3240
Recall ~0.77: threshold=0.5039, precision=0.3955
Recall ~0.75: threshold=0.5398, precision=0.4415
Recall ~0.72: threshold=0.5929, precision=0.5048
Recall ~0.70: threshold=0.6309, precision=0.5533
Recall ~0.68: threshold=0.6588, precision=0.5875
Recall ~0.65: threshold=0.7022, precision=0.6457

Best F1 score:
Threshold=0.7974, Recall=0.5729, Precision=0.7992, F1=0.6674


In [17]:
threshold = 0.5929
y_pred_new = (y_pred_proba > threshold).astype(int)

recall_new = recall_score(y_val, y_pred_new)
precision_new = precision_score(y_val, y_pred_new)
f1_new = f1_score(y_val, y_pred_new)

print(f"Threshold: {threshold}")
print(f"Recall: {recall_new:.4f}")
print(f"Precision: {precision_new:.4f}")
print(f"F1 Score: {f1_new:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_new))

Threshold: 0.5929
Recall: 0.7201
Precision: 0.5049
F1 Score: 0.5936

Confusion Matrix:
[[111057   2918]
 [  1157   2976]]


In [19]:
from sklearn.preprocessing import LabelEncoder

le_product = LabelEncoder()
le_card4 = LabelEncoder()

le_product.fit(df['ProductCD'])
le_card4.fit(df['card4'])



LabelEncoder()

In [22]:
feature_columns = X.columns.tolist()
print(f"Number of features: {len(feature_columns)}")
print(f"First 5 features: {feature_columns[:5]}")

Number of features: 357
First 5 features: ['card1_freq', 'card4_freq', 'product_freq', 'amt_dollars', 'amt_cents']


In [23]:
import joblib
import json

joblib.dump(model, 'fraud_detection_final.pkl')
joblib.dump(le_product, 'product_encoder.pkl')
joblib.dump(le_card4, 'card4_encoder.pkl')
joblib.dump(feature_columns, 'feature_columns.pkl')

threshold_config = {'threshold': 0.5929}
with open('threshold.json', 'w') as f:
    json.dump(threshold_config, f)

print("All 5 files saved successfully to /kaggle/working/")
print("- fraud_detection_final.pkl")
print("- product_encoder.pkl")
print("- card4_encoder.pkl")
print("- feature_columns.pkl")
print("- threshold.json")

All 5 files saved successfully to /kaggle/working/
- fraud_detection_final.pkl
- product_encoder.pkl
- card4_encoder.pkl
- feature_columns.pkl
- threshold.json


In [24]:
import zipfile
import os

files_to_zip = [
    'fraud_detection_final.pkl',
    'product_encoder.pkl',
    'card4_encoder.pkl',
    'feature_columns.pkl',
    'threshold.json'
]

with zipfile.ZipFile('final_model_files.zip', 'w') as zipf:
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file)
            print(f"Added: {file}")
        else:
            print(f"Warning: {file} not found")

print("\nCreated: final_model_files.zip")

Added: fraud_detection_final.pkl
Added: product_encoder.pkl
Added: card4_encoder.pkl
Added: feature_columns.pkl
Added: threshold.json

Created: final_model_files.zip


----